# Case 1 — Incremental Load at Terabyte Scale (Synthetic Databricks Demonstration)

**Assignment scenario:** 6 TB / ~9 billion SQL Server rows, ~25 million changes daily, updates up to 90 days, hard deletes, unreliable `ModifiedDate`.

**This executable demonstration uses 10,000,000 synthetic rows**, an immutable simulated CDC feed, chunked full load, replay-safe incremental Delta MERGE, a forced failure, restart, and exact reconciliation. It does **not** connect to a real SQL Server or prove production 6 TB performance. Run on Databricks Serverless with permission to create a schema and managed Delta tables.

In [0]:
%sql
SHOW CATALOGS;

catalog
samples
system
workspace


In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace.mj_case1;

In [0]:
from pyspark.sql import functions as F, Window
from pyspark.sql.types import DecimalType
from delta.tables import DeltaTable

CATALOG = "workspace"  
SCHEMA = "mj_case1"
N = 10_000_000
CHUNK_SIZE = 1_000_000
NUM_PARTITIONS = 32
UPDATE_COUNT = 100_000
DELETE_COUNT = 10_000
assert N >= UPDATE_COUNT + DELETE_COUNT
spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{CATALOG}`.`{SCHEMA}`")
spark.sql(f"USE CATALOG `{CATALOG}`")
spark.sql(f"USE SCHEMA `{SCHEMA}`")
def name(table): return f"`{CATALOG}`.`{SCHEMA}`.`{table}`"
print('Schema:', CATALOG, SCHEMA)

Schema: workspace mj_case1


## 1. Generate synthetic source history (10 million rows)

In [0]:
# Distributed Spark range:
source = (spark.range(1, N + 1, 1, numPartitions=NUM_PARTITIONS)
 .select(F.col('id').alias('transaction_id'))
 .withColumn('customer_id', (F.col('transaction_id') % 250_000 + 1).cast('long'))
 .withColumn('transaction_date', F.date_sub(F.lit('2026-10-31').cast('date'), (F.col('transaction_id') % 365).cast('int')))
 .withColumn('amount', ((F.col('transaction_id') % 100_000 + 100) / F.lit(100)).cast('decimal(18,2)'))
 .withColumn('status', F.lit('POSTED'))
 .withColumn('modified_date', F.to_timestamp(F.lit('2026-10-31 00:00:00'))))
source.write.format('delta').mode('overwrite').option('overwriteSchema','true').saveAsTable(name('source_transactions'))
print('Baseline rows:', spark.table(name('source_transactions')).count())

Baseline rows: 10000000


## 2. Parallel chunked full load with a durable chunk manifest

ID-range predicates are analogous to SQL Server partitioned JDBC extraction (`partitionColumn=transaction_id`, lower/upper bounds, `numPartitions`). Here the source is a managed synthetic Delta table. Each chunk is committed separately and tracked in a manifest. On restart, only unfinished chunks are retried. **In a real SQL Server full load, use a consistent snapshot or CDC LSN boundary** so rows changing during extraction are not lost.

In [0]:
spark.sql(f"DROP TABLE IF EXISTS {name('target_transactions')}")
spark.sql(f"DROP TABLE IF EXISTS {name('full_load_manifest')}")
spark.sql(f"CREATE TABLE {name('full_load_manifest')} (chunk_id BIGINT, low_id BIGINT, high_id BIGINT, row_count BIGINT, status STRING) USING DELTA")
# Create target with the source schema but no rows.
spark.table(name('source_transactions')).limit(0).write.format('delta').mode('overwrite').saveAsTable(name('target_transactions'))

def load_chunk(chunk_id):
    low = chunk_id * CHUNK_SIZE + 1
    high = min((chunk_id + 1) * CHUNK_SIZE, N)
    already = (spark.table(name('full_load_manifest'))
        .filter((F.col('chunk_id') == chunk_id) & (F.col('status') == 'DONE')).limit(1).count())
    if already:
        return 'SKIPPED'
    batch = spark.table(name('source_transactions')).filter(F.col('transaction_id').between(low, high))
    # MERGE ensures a retry after a crash between target commit and manifest commit is safe.
    (DeltaTable.forName(spark, name('target_transactions')).alias('t')
     .merge(batch.alias('s'), 't.transaction_id = s.transaction_id')
     .whenNotMatchedInsertAll().execute())
    actual = spark.table(name('target_transactions')).filter(F.col('transaction_id').between(low,high)).count()
    assert actual == high - low + 1
    manifest_row = spark.createDataFrame([(int(chunk_id), int(low), int(high), int(actual), 'DONE')],
                         'chunk_id long, low_id long, high_id long, row_count long, status string')
    (DeltaTable.forName(spark, name('full_load_manifest')).alias('m')
      .merge(manifest_row.alias('s'), 'm.chunk_id = s.chunk_id')
      .whenMatchedUpdateAll().whenNotMatchedInsertAll().execute())
    return 'LOADED'

num_chunks = (N + CHUNK_SIZE - 1) // CHUNK_SIZE
for chunk_id in range(num_chunks):
    print(f'Chunk {chunk_id+1}/{num_chunks}:', load_chunk(chunk_id))
assert spark.table(name('target_transactions')).count() == N

Chunk 1/10: LOADED
Chunk 2/10: LOADED
Chunk 3/10: LOADED
Chunk 4/10: LOADED
Chunk 5/10: LOADED
Chunk 6/10: LOADED
Chunk 7/10: LOADED
Chunk 8/10: LOADED
Chunk 9/10: LOADED
Chunk 10/10: LOADED


**Parallelization note:** The 10 million records are divided into smaller chunks. Spark processes the data using multiple partitions. The notebook loads chunks one at a time to reduce costs.

In production, multiple chunks can be loaded in parallel using Databricks Workflows or Azure Data Factory to improve performance.

## 3. Simulate an immutable CDC feed with updates and hard deletes

CDC is ordered by a monotonically increasing `change_sequence`, **not** `modified_date`. Duplicate events are deliberately included. Some updates leave `modified_date` unchanged.

In [0]:
base = spark.table(name('source_transactions'))
updates = (base.filter(F.col('transaction_id').between(1, UPDATE_COUNT))
 .withColumn('amount', (F.col('amount') + F.lit(10)).cast('decimal(18,2)'))
 .withColumn('status', F.lit('ADJUSTED'))
 .withColumn('modified_date', F.when(F.col('transaction_id') % 10 == 0,
                                    F.col('modified_date')).otherwise(F.to_timestamp(F.lit('2026-11-01 00:00:00'))))
 .withColumn('operation', F.lit('U'))
 .withColumn('change_sequence', F.col('transaction_id') + F.lit(N))
 .withColumn('change_timestamp', F.to_timestamp(F.lit('2026-11-01 01:00:00'))))
deletes = (base.filter(F.col('transaction_id').between(UPDATE_COUNT + 1, UPDATE_COUNT + DELETE_COUNT))
 .withColumn('operation', F.lit('D'))
 .withColumn('change_sequence', F.col('transaction_id') + F.lit(N))
 .withColumn('change_timestamp', F.to_timestamp(F.lit('2026-11-01 02:00:00'))))
# A repeated event for the same transaction tests source deduplication.
repeat = updates.filter(F.col('transaction_id') <= 1000)
cdc = updates.unionByName(deletes).unionByName(repeat)
cdc.write.format('delta').mode('overwrite').option('overwriteSchema','true').saveAsTable(name('synthetic_cdc_events'))
print('CDC rows including deliberate duplicates:', spark.table(name('synthetic_cdc_events')).count())
print('Distinct changed transaction IDs:', cdc.select('transaction_id').distinct().count())

CDC rows including deliberate duplicates: 111000
Distinct changed transaction IDs: 110000


In [0]:
display(spark.table(name('synthetic_cdc_events')))

transaction_id,customer_id,transaction_date,amount,status,modified_date,operation,change_sequence,change_timestamp
1,2,2026-10-30,11.01,ADJUSTED,2026-11-01T00:00:00.000Z,U,10000001,2026-11-01T01:00:00.000Z
2,3,2026-10-29,11.02,ADJUSTED,2026-11-01T00:00:00.000Z,U,10000002,2026-11-01T01:00:00.000Z
3,4,2026-10-28,11.03,ADJUSTED,2026-11-01T00:00:00.000Z,U,10000003,2026-11-01T01:00:00.000Z
4,5,2026-10-27,11.04,ADJUSTED,2026-11-01T00:00:00.000Z,U,10000004,2026-11-01T01:00:00.000Z
5,6,2026-10-26,11.05,ADJUSTED,2026-11-01T00:00:00.000Z,U,10000005,2026-11-01T01:00:00.000Z
6,7,2026-10-25,11.06,ADJUSTED,2026-11-01T00:00:00.000Z,U,10000006,2026-11-01T01:00:00.000Z
7,8,2026-10-24,11.07,ADJUSTED,2026-11-01T00:00:00.000Z,U,10000007,2026-11-01T01:00:00.000Z
8,9,2026-10-23,11.08,ADJUSTED,2026-11-01T00:00:00.000Z,U,10000008,2026-11-01T01:00:00.000Z
9,10,2026-10-22,11.09,ADJUSTED,2026-11-01T00:00:00.000Z,U,10000009,2026-11-01T01:00:00.000Z
10,11,2026-10-21,11.10,ADJUSTED,2026-10-31T00:00:00.000Z,U,10000010,2026-11-01T01:00:00.000Z


## 4. Build authoritative expected current state independently

We create an independent expected-state table to validate the Delta MERGE. After applying updates and deletes, we compare the expected table with the target to ensure there are no missing, duplicate, or incorrect records.

In [0]:
expected = (base.filter(~F.col('transaction_id').between(UPDATE_COUNT + 1, UPDATE_COUNT + DELETE_COUNT))
 .withColumn('amount', F.when(F.col('transaction_id') <= UPDATE_COUNT,
                              (F.col('amount') + F.lit(10)).cast('decimal(18,2)')).otherwise(F.col('amount')))
 .withColumn('status', F.when(F.col('transaction_id') <= UPDATE_COUNT, F.lit('ADJUSTED')).otherwise(F.col('status')))
 .withColumn('modified_date', F.when((F.col('transaction_id') <= UPDATE_COUNT) & (F.col('transaction_id') % 10 != 0),
                                    F.to_timestamp(F.lit('2026-11-01 00:00:00'))).otherwise(F.col('modified_date'))))
expected.write.format('delta').mode('overwrite').option('overwriteSchema','true').saveAsTable(name('source_current_state'))
print('Expected current rows:', spark.table(name('source_current_state')).count())

Expected current rows: 9990000


## 5. Replay-safe incremental MERGE

Process two durable sequence windows. Deduplicate by `transaction_id` before MERGE, taking the latest `change_sequence`; for equal sequence, delete wins. Replaying the same window is safe. Real SQL Server CDC must preserve commit LSN and operation ordering, including multiple changes per key.

In [0]:
spark.sql(f"DROP TABLE IF EXISTS {name('cdc_batch_manifest')}")
spark.sql(f"CREATE TABLE {name('cdc_batch_manifest')} (batch_id STRING, status STRING, applied_count BIGINT) USING DELTA")

def apply_cdc_batch(batch_id, low_id, high_id, simulate_failure=False):
    # Only process a committed, immutable CDC range. A rerun after the MERGE but before
    # checkpoint commit is safe because operations use absolute values and hard deletes.
    done = (spark.table(name('cdc_batch_manifest'))
            .filter((F.col('batch_id') == batch_id) & (F.col('status') == 'DONE')).limit(1).count())
    if done:
        print(batch_id, 'already DONE; skipping')
        return
    events = spark.table(name('synthetic_cdc_events')).filter(F.col('transaction_id').between(low_id, high_id))
    w = Window.partitionBy('transaction_id').orderBy(F.col('change_sequence').desc(),
                                                        F.when(F.col('operation') == 'D',1).otherwise(0).desc(),
                                                        F.col('change_timestamp').desc())
    dedup = (events.withColumn('rn', F.row_number().over(w)).filter('rn = 1').drop('rn'))
    assert dedup.groupBy('transaction_id').count().filter('count > 1').count() == 0
    (DeltaTable.forName(spark, name('target_transactions')).alias('t')
      .merge(dedup.alias('s'), 't.transaction_id = s.transaction_id')
      .whenMatchedDelete(condition="s.operation = 'D'")
      .whenMatchedUpdate(condition="s.operation = 'U'", set={
          'customer_id':'s.customer_id', 'transaction_date':'s.transaction_date',
          'amount':'s.amount', 'status':'s.status', 'modified_date':'s.modified_date'})
      .whenNotMatchedInsert(condition="s.operation <> 'D'", values={
          'transaction_id':'s.transaction_id', 'customer_id':'s.customer_id',
          'transaction_date':'s.transaction_date', 'amount':'s.amount',
          'status':'s.status', 'modified_date':'s.modified_date'})
      .execute())
    if simulate_failure:
        raise RuntimeError('INTENTIONAL FAILURE: target MERGE committed, checkpoint not written')
    applied = dedup.count()
    row = spark.createDataFrame([(batch_id, 'DONE', applied)], 'batch_id string, status string, applied_count long')
    (DeltaTable.forName(spark, name('cdc_batch_manifest')).alias('t')
      .merge(row.alias('s'), 't.batch_id = s.batch_id')
      .whenMatchedUpdateAll().whenNotMatchedInsertAll().execute())
    print(batch_id, 'DONE:', applied, 'distinct keys')

# First batch completes normally.
apply_cdc_batch('batch_1', 1, 50_000)
# Second batch fails AFTER committing the Delta MERGE but BEFORE recording completion.
try:
    apply_cdc_batch('batch_2', 50_001, UPDATE_COUNT + DELETE_COUNT, simulate_failure=True)
except RuntimeError as e:
    print('Expected test failure:', e)
# Restart replays exactly the same immutable CDC range.
apply_cdc_batch('batch_2', 50_001, UPDATE_COUNT + DELETE_COUNT)
# Additional retry must be a no-op.
apply_cdc_batch('batch_2', 50_001, UPDATE_COUNT + DELETE_COUNT)

batch_1 DONE: 50000 distinct keys
Expected test failure: INTENTIONAL FAILURE: target MERGE committed, checkpoint not written
batch_2 DONE: 60000 distinct keys
batch_2 already DONE; skipping


## 6. Exact reconciliation and restart assertions

In [0]:
target = spark.table(name('target_transactions'))
truth = spark.table(name('source_current_state'))
actual_count = target.count()
expected_count = truth.count()
assert actual_count == expected_count == N - DELETE_COUNT
assert target.groupBy('transaction_id').count().filter('count > 1').count() == 0
# EXCEPT ALL in both directions checks all columns and multiplicities (not only counts).
cols = ['transaction_id','customer_id','transaction_date','amount','status','modified_date']
missing = truth.select(*cols).exceptAll(target.select(*cols)).count()
extra = target.select(*cols).exceptAll(truth.select(*cols)).count()
assert missing == 0 and extra == 0, f'Mismatch: missing={missing}, extra={extra}'
assert spark.table(name('cdc_batch_manifest')).filter("status = 'DONE'").count() == 2
print({'baseline':N, 'updates':UPDATE_COUNT, 'deletes':DELETE_COUNT,
       'expected_target_rows':expected_count, 'actual_target_rows':actual_count,
       'missing_rows':missing, 'extra_rows':extra, 'restart_test':'PASS'})

{'baseline': 10000000, 'updates': 100000, 'deletes': 10000, 'expected_target_rows': 9990000, 'actual_target_rows': 9990000, 'missing_rows': 0, 'extra_rows': 0, 'restart_test': 'PASS'}
